# 面试问题：长时间运行的 Agent 怎样做到崩溃恢复、精确重放和安全取消？

**一句话回答**：把 Agent Loop 视为耐久状态机，以 append-only 事件记录 proposed/started/completed/failed/waiting/cancelled；纯 reducer 从事件恢复状态，外部副作用用幂等键和结果账本；周期快照加速恢复但事件仍是真相。重放不能重新调用模型/工具，而应使用已记录结果；版本升级通过显式 state migration。

本 Notebook 实现 hash-chain 事件日志、reducer、快照、幂等副作用、确定性退避、HITL 等待、取消补偿和 schema 迁移。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import copy, hashlib, json, math  # 导入本单元所需的依赖。

SEED119=11901; GENESIS119="0"*64  # 计算并保存当前步骤的中间状态。
assert SEED119==11901  # 用受控断言验证关键不变量。
assert len(GENESIS119)==64  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"event").hexdigest()!=GENESIS119  # 用受控断言验证关键不变量。

## 1. 事件是不可变事实，状态是可重建视图

每条事件有 run、递增 seq、类型、payload、前序 hash 和自身 hash。写入时验证 expected seq，防止两个 worker 同时追加。不要把完整敏感 Prompt 无条件落日志，事件 payload 只存必要摘要或加密引用。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Event119: run_id:str; seq:int; kind:str; payload:dict; prev_hash:str; event_hash:str  # 定义承载本节状态与行为的数据结构。
log119=[]  # 计算并保存当前步骤的中间状态。
def append119(run_id,kind,payload,expected_seq):  # 定义本节可复用的核心函数。
    if expected_seq!=len(log119): raise ValueError("sequence_conflict")  # 按当前条件选择后续控制路径。
    prev=log119[-1].event_hash if log119 else GENESIS119; raw=json.dumps({"run":run_id,"seq":expected_seq,"kind":kind,"payload":payload,"prev":prev},sort_keys=True,separators=(",",":")); h=hashlib.sha256(raw.encode()).hexdigest(); e=Event119(run_id,expected_seq,kind,copy.deepcopy(payload),prev,h); log119.append(e); return e  # 计算并保存当前步骤的中间状态。
e0_119=append119("r1","run_started",{"goal":"report"},0)  # 计算并保存当前步骤的中间状态。
assert e0_119.seq==0 and e0_119.prev_hash==GENESIS119  # 用受控断言验证关键不变量。
assert len(e0_119.event_hash)==64  # 用受控断言验证关键不变量。
try: append119("r1","x",{},0); raise AssertionError("stale append accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="sequence_conflict"  # 捕获预期异常并验证失败分支。

## 2. Reducer 必须纯且确定

reducer 只依据旧状态和事件计算新状态，不读当前时间、随机数或网络。相同事件序列在新进程中重放得到同一状态。未知事件类型失败关闭，不能静默忽略导致状态含义变化。

In [ ]:
def initial119(): return {"status":"new","goal":None,"completed":{},"waiting":None,"cancelled":False,"last_seq":-1}  # 定义本节可复用的核心函数。
def reduce119(state,event):  # 定义本节可复用的核心函数。
    s=copy.deepcopy(state)  # 计算并保存当前步骤的中间状态。
    if event.seq!=s["last_seq"]+1: raise ValueError("replay_sequence")  # 按当前条件选择后续控制路径。
    if event.kind=="run_started": s["status"]="running"; s["goal"]=event.payload["goal"]  # 按当前条件选择后续控制路径。
    elif event.kind=="step_completed": s["completed"][event.payload["step_id"]]=event.payload["result"]  # 按当前条件选择后续控制路径。
    elif event.kind=="waiting": s["status"]="waiting"; s["waiting"]=event.payload  # 按当前条件选择后续控制路径。
    elif event.kind=="resumed": s["status"]="running"; s["waiting"]=None  # 按当前条件选择后续控制路径。
    elif event.kind=="cancelled": s["status"]="cancelled"; s["cancelled"]=True  # 按当前条件选择后续控制路径。
    elif event.kind=="run_completed": s["status"]="completed"  # 按当前条件选择后续控制路径。
    else: raise ValueError("unknown_event")  # 执行当前语句以推进本节示例。
    s["last_seq"]=event.seq; return s  # 计算并保存当前步骤的中间状态。
state119=reduce119(initial119(),e0_119)  # 计算并保存当前步骤的中间状态。
assert state119["status"]=="running" and state119["goal"]=="report"  # 用受控断言验证关键不变量。
assert state119["last_seq"]==0  # 用受控断言验证关键不变量。
assert initial119()["status"]=="new"  # 用受控断言验证关键不变量。

## 3. 重放验证 hash chain，不重新执行外部动作

Replay 先验证 prev hash 与 event hash，再调用 reducer。模型响应和工具结果必须作为 completed 事件记录；恢复时直接注入记录结果，若重新请求会产生不同输出或重复副作用。日志损坏应停止并告警。

In [ ]:
append119("r1","step_completed",{"step_id":"search","result":{"docs":[1,2]}},1)  # 执行当前语句以推进本节示例。
def replay119(events):  # 定义本节可复用的核心函数。
    state=initial119(); prev=GENESIS119  # 计算并保存当前步骤的中间状态。
    for e in events:  # 遍历输入元素以累积或检查结果。
        raw=json.dumps({"run":e.run_id,"seq":e.seq,"kind":e.kind,"payload":e.payload,"prev":e.prev_hash},sort_keys=True,separators=(",",":")); expected=hashlib.sha256(raw.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
        if e.prev_hash!=prev or e.event_hash!=expected: raise ValueError("event_integrity")  # 按当前条件选择后续控制路径。
        state=reduce119(state,e); prev=e.event_hash  # 计算并保存当前步骤的中间状态。
    return state  # 返回当前分支计算出的结果。
replayed119=replay119(log119)  # 计算并保存当前步骤的中间状态。
assert replayed119["completed"]["search"]=={"docs":[1,2]}  # 用受控断言验证关键不变量。
assert replayed119["last_seq"]==1  # 用受控断言验证关键不变量。
assert replay119(log119)==replay119(log119)  # 用受控断言验证关键不变量。

## 4. Snapshot 加速恢复，但不能替代日志

在某 seq 保存 reducer state 与对应 event hash，恢复时校验后只重放后续事件。snapshot 写临时文件再原子替换；若 snapshot 损坏，可回退到旧 snapshot + log。schema version 用于迁移。

In [ ]:
snapshot119={"schema":1,"seq":replayed119["last_seq"],"event_hash":log119[-1].event_hash,"state":copy.deepcopy(replayed119)}  # 计算并保存当前步骤的中间状态。
append119("r1","waiting",{"reason":"approval","action":"publish"},2)  # 执行当前语句以推进本节示例。
def restore119(snapshot,events):  # 定义本节可复用的核心函数。
    if snapshot["event_hash"]!=events[snapshot["seq"]].event_hash: raise ValueError("snapshot_mismatch")  # 按当前条件选择后续控制路径。
    state=copy.deepcopy(snapshot["state"])  # 计算并保存当前步骤的中间状态。
    for e in events[snapshot["seq"]+1:]: state=reduce119(state,e)  # 遍历输入元素以累积或检查结果。
    return state  # 返回当前分支计算出的结果。
restored119=restore119(snapshot119,log119)  # 计算并保存当前步骤的中间状态。
assert restored119["status"]=="waiting" and restored119["waiting"]["action"]=="publish"  # 用受控断言验证关键不变量。
assert restored119["last_seq"]==2  # 用受控断言验证关键不变量。
assert snapshot119["state"]["status"]=="running"  # 用受控断言验证关键不变量。

## 5. 外部副作用由幂等账本跨崩溃保护

在调用前记录 intent，使用稳定 idempotency key；返回后记录结果。若崩溃发生在远端提交后、本地 completed 前，恢复先向远端/账本 reconcile，不能盲重试。相同 key 不同参数必须冲突。

In [ ]:
side_effects119={}; ledger119={}  # 计算并保存当前步骤的中间状态。
def effect119(key,args):  # 定义本节可复用的核心函数。
    fp=hashlib.sha256(json.dumps(args,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
    if key in ledger119:  # 按当前条件选择后续控制路径。
        if ledger119[key]["fp"]!=fp: raise ValueError("idempotency_conflict")  # 按当前条件选择后续控制路径。
        return ledger119[key]["result"],True  # 返回当前分支计算出的结果。
    result={"remote_id":f"pub-{len(side_effects119)+1}"}; side_effects119[key]=copy.deepcopy(args); ledger119[key]={"fp":fp,"result":result}; return result,False  # 计算并保存当前步骤的中间状态。
r1_119,replay_flag119=effect119("r1:publish",{"doc":"v3"}); r2_119,replay_flag2_119=effect119("r1:publish",{"doc":"v3"})  # 计算并保存当前步骤的中间状态。
assert r1_119==r2_119 and not replay_flag119 and replay_flag2_119  # 用受控断言验证关键不变量。
assert len(side_effects119)==1  # 用受控断言验证关键不变量。
try: effect119("r1:publish",{"doc":"v4"}); raise AssertionError("conflict accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="idempotency_conflict"  # 捕获预期异常并验证失败分支。

## 6. Retry 时间也写成确定事件

退避使用 attempt 计算，随机 jitter 需在首次决定时记录，而不是重放时重新采样。Timer 触发是事件；恢复后若到期立即处理，未到期重新注册。全局 deadline 优先于节点重试。

In [ ]:
def retry_delay119(attempt,base=2,cap=30,recorded_jitter=0): return min(cap,base*(2**attempt))+recorded_jitter  # 定义本节可复用的核心函数。
delays119=[retry_delay119(i,recorded_jitter=.25) for i in range(5)]  # 计算并保存当前步骤的中间状态。
assert delays119==[2.25,4.25,8.25,16.25,30.25]  # 用受控断言验证关键不变量。
assert retry_delay119(3,recorded_jitter=.25)==retry_delay119(3,recorded_jitter=.25)  # 用受控断言验证关键不变量。
assert all(a<b for a,b in zip(delays119[:4],delays119[1:4]))  # 用受控断言验证关键不变量。

## 7. HITL 等待、取消和补偿都是状态转移

waiting 状态不占 worker；审批 signal 带 action digest 和预期 seq，防止旧批准恢复新动作。取消只阻止未来步骤，已提交动作通过 saga 逆序补偿；不可逆动作在执行前必须审批。

In [ ]:
append119("r1","resumed",{"approval":"ok"},3); append119("r1","step_completed",{"step_id":"publish","result":r1_119},4); append119("r1","run_completed",{},5)  # 执行当前语句以推进本节示例。
final119=replay119(log119)  # 计算并保存当前步骤的中间状态。
compensations119={"reserve":"release","publish":"unpublish"}  # 计算并保存当前步骤的中间状态。
def compensation_plan119(completed_order): return [compensations119[x] for x in reversed(completed_order) if x in compensations119]  # 定义本节可复用的核心函数。
assert final119["status"]=="completed" and final119["waiting"] is None  # 用受控断言验证关键不变量。
assert final119["completed"]["publish"]=={"remote_id":"pub-1"}  # 用受控断言验证关键不变量。
assert compensation_plan119(["search","reserve","publish"])==["unpublish","release"]  # 用受控断言验证关键不变量。

## 8. 代码升级必须迁移旧状态并保持可重放

新 reducer 不能改变旧事件含义；按 event/schema version 分支或先离线迁移快照。部署前用历史 trace replay 比较最终状态，rolling upgrade 期间 worker 只领取兼容版本任务。manifest 固定代码、模型、工具和事件 schema。

In [ ]:
def migrate_snapshot119(snap):  # 定义本节可复用的核心函数。
    out=copy.deepcopy(snap)  # 计算并保存当前步骤的中间状态。
    if out["schema"]==1: out["state"].setdefault("cost",0.); out["schema"]=2  # 按当前条件选择后续控制路径。
    return out  # 返回当前分支计算出的结果。
migrated119=migrate_snapshot119(snapshot119); manifest119={"schema":2,"event_schema":"agent-events-v2","reducer":"sha-demo-v3","snapshot_every":50,"replay_external_calls":False,"side_effects":"idempotency+reconcile","cancellation":"saga"}; digest119=hashlib.sha256(json.dumps(manifest119,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert migrated119["schema"]==2 and migrated119["state"]["cost"]==0  # 用受控断言验证关键不变量。
assert snapshot119["schema"]==1 and not manifest119["replay_external_calls"]  # 用受控断言验证关键不变量。
assert len(digest119)==64  # 用受控断言验证关键不变量。

## 面试总结

主线是：**append-only hash-chain events → 纯 reducer → completed 结果重放 → snapshot → 幂等/对账副作用 → 记录化 retry timer → HITL signal/cancel/saga → schema migration 与历史 trace 回归**。耐久执行保证进程失败后继续同一逻辑，不等同于“所有业务恰好一次”。

延伸阅读：[Temporal Durable Execution 概念](https://docs.temporal.io/temporal)、[Event Sourcing](https://martinfowler.com/eaaDev/EventSourcing.html)、[OWASP Excessive Agency](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/)。